In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Layer 2: Intermediate ESI Random Forest & Hierarchical Cascade with Feature Engineering (`models/rf_feng_extreme.ipynb`)

This notebook trains the **Layer 2 Feature Engineered Model** and evaluates the **2-Layer Feature Engineered Hierarchical Cascade System**:
- **Layer 2 Model**: Random Forest trained on Feature Engineered inputs + `age` + `gender`, **excluding rows containing ESI 1 and 5** (only ESI 2, ESI 3, ESI 4).
- **Input Features**: `age`, `gender`, and the **10 Clinical Feature Engineering flags** defined in `TODO.md` (`is_dyspnea_total`, `is_dyspnea_moderate`, `is_bradypnea`, `is_tachypnea`, `is_hypotension`, `is_hypertension`, `is_bradycardia_total`, `is_bradycardia_moderate`, `is_tachycardia_total`, `is_tachycardia_moderate`).
- **Hierarchical Cascade Flow**:
  - Layer 1 (Feature Engineered Logistic Regression) evaluates sample: returns ESI 1, ESI 5, or 'neither'.
  - If 'neither', Layer 2 (Feature Engineered Random Forest) evaluates sample: decides ESI 2, ESI 3, or ESI 4.
- **Scoring Metrics**: **Accuracy**, **ROC-AUC**, and **Log Loss**.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
library(jsonlite)
library(caret)
library(ranger)
library(nnet)
library(dplyr)
library(ggplot2)
library(pROC)

config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}

config <- fromJSON(config_path)

cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Val Size:        ", config$training$val_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Data, Compute Feature Engineered Inputs, & Filter Out ESI 1 & 5
# ---------------------------------------------------------
set.seed(config$training$random_state)

data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}

cat("Loading dataset from:", data_file, "...\n")

data_env <- new.env()
load(data_file, envir = data_env)

df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
cat(sprintf("Selected main dataset object: '%s' (%d rows)\n", data_obj_name, max(df_sizes)))

raw_df <- get(data_obj_name, envir = data_env)
target_col <- config$classes$target_col

gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0

# Compute 10 Clinical Feature Engineering flags + Age + Gender
df_feng <- data.frame(
  age                     = raw_df$age,
  gender                  = gender_vec,
  is_dyspnea_total        = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 < 90, 1, 0),
  is_dyspnea_moderate     = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 >= 90 & raw_df$triage_vital_o2 < 94, 1, 0),
  is_bradypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr < 10, 1, 0),
  is_tachypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr > 30, 1, 0),
  is_hypotension          = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp <= 90, 1, 0),
  is_hypertension         = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp > 220, 1, 0),
  is_bradycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr < 40, 1, 0),
  is_bradycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr >= 40 & raw_df$triage_vital_hr < 60, 1, 0),
  is_tachycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 150, 1, 0),
  is_tachycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 100 & raw_df$triage_vital_hr <= 150, 1, 0)
)
df_feng[[target_col]] <- raw_df[[target_col]]

# Filter dataset for Layer 2: REMOVE ESI 1 and ESI 5
raw_esi_char <- as.character(df_feng[[target_col]])
df_layer2 <- df_feng[raw_esi_char %in% c("2", "3", "4"), ]
df_layer2[[target_col]] <- factor(as.character(df_layer2[[target_col]]), levels = c("2", "3", "4"))

if (any(is.na(df_layer2))) {
  df_layer2 <- na.omit(df_layer2)
}

cat(sprintf("Layer 2 Filtered Dataset Ready (No ESI 1 & 5): %d rows x %d cols\n", nrow(df_layer2), ncol(df_layer2)))
cat("Layer 2 Target Distribution:\n")
print(table(df_layer2[[target_col]]))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Stratified Data Partitioning for Layer 2
# ---------------------------------------------------------
set.seed(config$training$random_state)

test_size <- config$training$test_size
val_size  <- config$training$val_size

in_train_val <- createDataPartition(df_layer2[[target_col]], p = 1 - test_size, list = FALSE)
train_val_df <- df_layer2[in_train_val, ]
test_df_l2   <- df_layer2[-in_train_val, ]

rel_val_size <- val_size / (1 - test_size)
in_train    <- createDataPartition(train_val_df[[target_col]], p = 1 - rel_val_size, list = FALSE)
train_df_l2 <- train_val_df[in_train, ]
val_df_l2   <- train_val_df[-in_train, ]

cat(sprintf("Layer 2 Partitions:\n  Train: %d rows\n  Val:   %d rows\n  Test:  %d rows\n",
            nrow(train_df_l2), nrow(val_df_l2), nrow(test_df_l2)))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Train Layer 2 Feature Engineered Random Forest Model
# ---------------------------------------------------------
set.seed(config$training$random_state)

formula_rf <- as.formula(paste(target_col, "~ ."))

cat("Training Layer 2 Feature Engineered Random Forest on ESI 2, 3, 4 data...\n")
rf_feng_layer2 <- ranger(
  formula = formula_rf,
  data = train_df_l2,
  num.trees = 300,
  importance = "impurity",
  probability = TRUE,
  seed = config$training$random_state
)

cat("Layer 2 Feature Engineered Random Forest training complete!\n")
print(rf_feng_layer2)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Standalone Layer 2 Benchmark (Accuracy, ROC-AUC, Log Loss)
# ---------------------------------------------------------

calc_log_loss <- function(actual_factor, prob_matrix, eps = 1e-15) {
  prob_matrix <- pmax(pmin(prob_matrix, 1 - eps), eps)
  prob_matrix <- prob_matrix / rowSums(prob_matrix)
  classes <- colnames(prob_matrix)
  N <- length(actual_factor)
  
  log_probs <- numeric(N)
  for (i in 1:N) {
    act_cls <- as.character(actual_factor[i])
    if (act_cls %in% classes) {
      log_probs[i] <- log(prob_matrix[i, act_cls])
    } else {
      log_probs[i] <- log(eps)
    }
  }
  return(-mean(log_probs))
}

evaluate_layer2_rf <- function(model, data, set_name, target_col) {
  prob_matrix <- predict(model, data = data)$predictions
  target_classes <- levels(data[[target_col]])
  
  max_idx <- max.col(prob_matrix, ties.method = "first")
  pred_factor <- factor(colnames(prob_matrix)[max_idx], levels = target_classes)
  actual_factor <- factor(data[[target_col]], levels = target_classes)
  
  cm <- confusionMatrix(pred_factor, actual_factor)
  acc <- as.numeric(cm$overall["Accuracy"])
  
  roc_auc <- tryCatch({
    as.numeric(pROC::multiclass.roc(actual_factor, prob_matrix)$auc)
  }, error = function(e) NA)
  
  log_loss <- calc_log_loss(actual_factor, prob_matrix)
  
  cat(sprintf("============================================================\n"))
  cat(sprintf("   STANDALONE LAYER 2 FEATURE ENGINEERED RF - %s SET BENCHMARK\n", toupper(set_name)))
  cat(sprintf("============================================================\n"))
  cat(sprintf("  Accuracy   : %.4f (%.2f%%)\n", acc, acc * 100))
  cat(sprintf("  ROC-AUC    : %.4f\n", roc_auc))
  cat(sprintf("  Log Loss   : %.4f\n", log_loss))
  cat("\nConfusion Matrix:\n")
  print(cm$table)
  cat(sprintf("============================================================\n\n"))
}

# Standalone Layer 2 FE Evaluations
evaluate_layer2_rf(rf_feng_layer2, val_df_l2, "Validation", target_col)
evaluate_layer2_rf(rf_feng_layer2, test_df_l2, "Test", target_col)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Full 2-Layer Feature Engineered Hierarchical Cascade Evaluation
# ---------------------------------------------------------

# Load Layer 1 Feature Engineered Model
lr_path <- file.path("../deploy", "lr_feng_extreme_model.rds")
if (!file.exists(lr_path)) lr_path <- file.path("deploy", "lr_feng_extreme_model.rds")

if (file.exists(lr_path)) {
  lr_obj <- readRDS(lr_path)
  cat("Successfully loaded Layer 1 Feature Engineered LR model from:", lr_path, "\n")
} else {
  cat("Layer 1 FE model not found at:", lr_path, ". Training quick inline Layer 1 FE model...\n")
  df_l1 <- df_feng
  raw_esi_c <- as.character(df_l1[[target_col]])
  df_l1$target_layer1 <- factor(ifelse(raw_esi_c == "1", "1", ifelse(raw_esi_c == "5", "5", "neither")), levels = c("1", "5", "neither"))
  preproc_l1 <- preProcess(df_l1[, "age", drop = FALSE], method = c("center", "scale"))
  df_l1$age <- predict(preproc_l1, df_l1[, "age", drop = FALSE])$age
  formula_l1 <- as.formula(paste("target_layer1 ~", paste(setdiff(names(df_l1), c(target_col, "target_layer1")), collapse = " + ")))
  lr_mod_inline <- multinom(formula_l1, data = df_l1, trace = FALSE)
  lr_obj <- list(model = lr_mod_inline, preproc = preproc_l1)
}

# Predict Function for Feature Engineered 2-Layer Cascade System
predict_hierarchical_cascade <- function(lr_obj, rf_model, newdata, target_col) {
  N <- nrow(newdata)
  target_classes_5 <- c("1", "2", "3", "4", "5")
  
  # 1. Scale age for Layer 1
  data_l1 <- newdata
  if (!is.null(lr_obj$preproc) && "age" %in% names(data_l1)) {
    data_l1[, "age"] <- predict(lr_obj$preproc, data_l1[, "age", drop = FALSE])$age
  }
  
  # 2. Layer 1 Probabilities (P(1), P(5), P(neither))
  prob_l1 <- predict(lr_obj$model, newdata = data_l1, type = "probs")
  if (!is.matrix(prob_l1)) {
    prob_l1 <- matrix(prob_l1, nrow = N, ncol = 3, byrow = TRUE)
    colnames(prob_l1) <- c("1", "5", "neither")
  }
  
  # 3. Layer 2 Probabilities (P(2), P(3), P(4))
  prob_l2 <- predict(rf_model, data = newdata)$predictions
  
  # 4. Construct Unified 5-Class Probability Matrix
  prob_5class <- matrix(0, nrow = N, ncol = 5)
  colnames(prob_5class) <- target_classes_5
  
  prob_5class[, "1"] <- prob_l1[, "1"]
  prob_5class[, "5"] <- prob_l1[, "5"]
  
  p_neither <- prob_l1[, "neither"]
  prob_5class[, "2"] <- p_neither * prob_l2[, "2"]
  prob_5class[, "3"] <- p_neither * prob_l2[, "3"]
  prob_5class[, "4"] <- p_neither * prob_l2[, "4"]
  
  # Normalize probabilities per sample
  prob_5class <- prob_5class / rowSums(prob_5class)
  
  max_idx <- max.col(prob_5class, ties.method = "first")
  pred_classes <- factor(target_classes_5[max_idx], levels = target_classes_5)
  
  return(list(predictions = pred_classes, probabilities = prob_5class))
}

evaluate_hierarchical_cascade <- function(lr_obj, rf_model, test_data, set_name, target_col) {
  res <- predict_hierarchical_cascade(lr_obj, rf_model, test_data, target_col)
  target_classes_5 <- c("1", "2", "3", "4", "5")
  actual_factor <- factor(as.character(test_data[[target_col]]), levels = target_classes_5)
  
  cm <- confusionMatrix(res$predictions, actual_factor)
  acc <- as.numeric(cm$overall["Accuracy"])
  
  roc_auc <- tryCatch({
    as.numeric(pROC::multiclass.roc(actual_factor, res$probabilities)$auc)
  }, error = function(e) NA)
  
  log_loss <- calc_log_loss(actual_factor, res$probabilities)
  
  cat(sprintf("============================================================\n"))
  cat(sprintf("   2-LAYER FEATURE ENGINEERED CASCADE SYSTEM (%s SET)\n", toupper(set_name)))
  cat(sprintf("============================================================\n"))
  cat(sprintf("  Overall Accuracy : %.4f (%.2f%%)\n", acc, acc * 100))
  cat(sprintf("  Multi-Class ROC-AUC : %.4f\n", roc_auc))
  cat(sprintf("  Log Loss            : %.4f\n", log_loss))
  cat("\nFull 5-Class Confusion Matrix:\n")
  print(cm$table)
  cat(sprintf("============================================================\n\n"))
}

# Evaluate Hierarchical Cascade on Full Dataset Test Split
in_train_val_full <- createDataPartition(df_feng[[target_col]], p = 1 - test_size, list = FALSE)
test_df_full <- df_feng[-in_train_val_full, ]

evaluate_hierarchical_cascade(lr_obj, rf_feng_layer2, test_df_full, "Full Test Split", target_col)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 7: Save Feature Engineered Model Artifacts
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)

saveRDS(rf_feng_layer2, file.path(deploy_dir, "rf_feng_extreme_model.rds"))
cat("Layer 2 Feature Engineered Random Forest saved to:", file.path(deploy_dir, "rf_feng_extreme_model.rds"), "\n")

cascade_obj <- list(lr_model = lr_obj, rf_model = rf_feng_layer2)
class(cascade_obj) <- "hierarchical_triage_feng_cascade"
saveRDS(cascade_obj, file.path(deploy_dir, "hierarchical_feng_cascade_model.rds"))
cat("Complete Feature Engineered Hierarchical Cascade System saved to:", file.path(deploy_dir, "hierarchical_feng_cascade_model.rds"), "\n")